# Data Cleaner

In [ ]:
import pandas as pd
import pyarrow as pa

import enchant
import re


pandas version: 3.0.1
pyarrow version: 23.0.1


In [29]:
data_path = './data/AUTokens50/part_0.parquet'

table = pq.read_table(data_path, use_pandas_metadata=False)
df = table.to_pandas()

In [38]:
df.head()

,text,id,dump,url,date,file_path,language,language_score,token_count
664,"Justine Davies –, Monday, January, 31, 2011, (...",<urn:uuid:0f3c1f06-4af1-4da0-96f9-91a7473f205f>,CC-MAIN-2013-20,http://blogs.news.com.au/moneystuff/index.php/...,2013-05-18T06:19:58Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,en,0.961377,1484
5457,Interview: Jenny Macpherson\n- by Rowena Scott...,<urn:uuid:12bb0bca-ddf8-4522-97ee-5583b8c93b93>,CC-MAIN-2013-20,http://www.bicycles.net.au/2010/10/interview-j...,2013-05-18T06:53:36Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,en,0.967261,2409
5459,The foundations for successful riding\n19 post...,<urn:uuid:74766fcf-d5eb-44fd-b925-01bd98098d70>,CC-MAIN-2013-20,http://www.bicycles.net.au/forums/viewtopic.ph...,2013-05-18T06:29:13Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,en,0.975258,2439
8507,photos by Carlo Ledesma\nWhen God was creating...,<urn:uuid:161ec30a-6da3-4d83-be97-32d4d9684b0c>,CC-MAIN-2013-20,http://www.kluster.com.au/issuefive/bees/,2013-05-18T07:25:14Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,en,0.961999,1419
9656,Joanne Harris is apparently as formidable a Yo...,<urn:uuid:2058b3ba-a9ad-4d81-80ab-4ce6b79c5d9e>,CC-MAIN-2013-20,http://www.northerndailyleader.com.au/story/97...,2013-05-18T08:10:31Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,en,0.990345,1806


# Implementation Notes

It is hard to tell if an entry is Australian or US. However, there are fome features that can be used to distinguish between them two.

## Special Terms

There are terms that can be used to tell, for example:

- **AU**: ABN, TFN, GST, ATO...
- **US**: EIN, TIN, SSN, IRS...

Also, slangs can be used to tell the country as well.

There are many other terms as well, so we probably need some sort of dictionary.

In [ ]:
Legal_AU = ['ABN', 'TFN', 'GST', 'ATO']
Legal_US = ['EIN', 'TIN', 'SSN', 'IRS']

Slang_AU = ['Aussie', 'Roo', 'Oz', 'Barbie', 'Bogan',
    'Avro', 'Bikkie', 'Avo']

Slang_US = ['Sup', 'Yo']

term_AU = Legal_AU + Slang_AU
term_US = Legal_US + Slang_US

def is_term_au(word):
    """
    check if the word is a term in Australian English.
    """
    return word in term_AU

def is_term_us(word):
    """
    check if the word is a term in US English.
    """
    return word in term_US

### Conventions

The conventions can also tell the difference, however, it is not deterministic:

- **AU**:
    - Date format: DD-MM-YYYY
    - Metric system
- **US**:
    - Date format: MM-DD-YYYY
    - Miles, Fahrenheit


In [ ]:
au_date_pattern = r'\b(0[1-9]|[12][0-9]|3[01])[\-/](0[1-9]|1[0-2])[\-/](\d{4})\b'
us_date_pattern = r'\b(0[1-9]|1[0-2])[\-/](0[1-9]|[12][0-9]|3[01])[\-/](\d{4})\b'
metric_units = ['km', 'kg', 'celsius', 'centimetre',
    'centimeter', 'metre', 'meter', 'litre', 'liter']
imperial_units = ['miles', 'mph', 'fahrenheit', 'feet',
    'foot', 'inches', 'inch', 'pounds', 'lbs']

def if_date_au(date):
    """
    check if the date string is in Australian date format (DD-MM-YYYY)。
    """
    #  match DD-MM-YYYY or DD/MM/YYYY format
    au_pattern = r'^(0[1-9]|[12][0-9]|3[01])[\-/](0[1-9]|1[0-2])[\-/](\d{4})$'
    return bool(re.match(au_pattern, str(date).strip()))

### Language

The spelling and everyday vocabulary can also tell the difference, however, it is alo not deterministic.

In [ ]:
us_dict = enchant.Dict("en_US")
au_dict = enchant.Dict("en_AU")


## Method

So, it is clear that we can use all the features above. Therefore, we can propose a scoring system to give the confidence score to each entry.

In [ ]:
def get_text_score(text):
    """
    get the score of the text in Australian English .

    > 0 if the text is more likely in Australian English.
    < 0 if the text is more likely in US English.
    """

    score = 0

    # Convert text to lowercase for case-insensitive matching
    text_lower = text.lower()
    words = re.findall(r'\b\w+\b', text_lower)

    # 1. Check for Australian-specific terms (high confidence)
    au_terms_lower = [term.lower() for term in term_AU]
    for term in au_terms_lower:
        if term in text_lower:
            score += 2

    # 2. Check for US-specific terms (high confidence)
    us_terms_lower = [term.lower() for term in term_US]
    for term in us_terms_lower:
        if term in text_lower:
            score -= 2

    # 3. Check spelling differences using enchant dictionaries
    for word in words:
        if len(word) > 3:  # Only check words longer than 3 characters
            is_au = au_dict.check(word)
            is_us = us_dict.check(word)

            # Word is valid in AU but not in US
            if is_au and not is_us:
                score += 0.5
            # Word is valid in US but not in AU
            elif is_us and not is_au:
                score -= 0.5

    # 4. Check for Australian date patterns (DD-MM-YYYY or DD/MM/YYYY)
    au_dates = re.findall(au_date_pattern, text)
    score += len(au_dates) * 0.3

    # 5. Check for US date patterns (MM-DD-YYYY or MM/DD/YYYY)
    us_dates = re.findall(us_date_pattern, text)
    score -= len(us_dates) * 0.3

    # 6. Check for metric system indicators (AU)
    for unit in metric_units:
        if unit in text_lower:
            score += 0.2

    # 7. Check for imperial system indicators (US)
    for unit in imperial_units:
        if unit in text_lower:
            score -= 0.2

    return score